<span style="float: left;padding: 1.3em">![logo](../logo.png)</span>

# Gravitational Wave Open Data Workshop

## Tutorial 3.1: Fourier transforms

This tutorial presents the basic concepts of signal processing.

View this tutorial on [Google Colaboratory](https://colab.research.google.com/github/gw-odw/odw/blob/main/Tutorials/03_Signal_Processing/Tuto_3.1_Fourier_Transforms.ipynb) or launch [mybinder](https://mybinder.org/v2/gh/gw-odw/odw/HEAD).


## Installation (execute only if running on a cloud platform, like Google Colab, or if you haven't done the installation already!)

> ⚠️ **Warning**: restart the runtime after running the cell below.
>
> To do so, click "Runtime" in the menu and choose "Restart and run all".
>
> You may see error messages but installation usually works.
> If you experience problems, please [report an issue](https://github.com/gw-odw/odw/issues).

In [ ]:
# -- Uncomment the following line only if running in Google Colab
# -- or if the required packages are not already installed.
# ! pip install -q 'gwpy==3.0.14' 'cryptography==43.0.0' 'PyCBC==2.10.0'

## Initialization

In [ ]:
# Those 2 lines are just to avoid some harmless warnings when importing packages
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import pycbc
import gwpy

from ipywidgets import interact, interactive, fixed, interact_manual, widgets
from pycbc.waveform import get_td_waveform, get_fd_waveform

## Fourier transforms

The [Fourier transform](https://en.wikipedia.org/wiki/Fourier_transform) is a widely-used mathematical tool to expose the frequency-domain content of a time-domain signal $s(t)$, meaning we can see which frequencies contain lots of power, and which have less.

For a given frequency $f$, the Fourier transform of $s(t)$ is:
$$
\tilde{s}(f) = \int_{-\infty}^{+\infty} s(t)\, e^{- 2\pi i f t} dt
$$
The Fourier transform is a complex-valued function of $f$. For real-valued signals, positive and negative frequencies are redundant as $\tilde{s}(-f) = \tilde{s}^*(f)$.

Another useful property of Fourier transforms, that you can check on the definition: if $s_{\Delta T}(t) = s(t - \Delta T)$ is shifted in time by $+\Delta T$ from the signal $s$, then we have $ \tilde{s}_{\Delta T} (f) = e^{-2 i \pi f \Delta T} \tilde{s}(f) $

## The Discrete Fourier Transform: basics

We consider here signals as functions of time $s(t)$, which can be either deterministic or stochastic in nature.
When working with digital data, we have to sample this signal according to a discrete time grid, usually with a fixed interval $\Delta t$:
$$
s_j = s(j \times \Delta t)
$$
$f_s = \frac{1}{\Delta t}$ is called the sampling frequency.

The Discrete Fourier Transform (DFT) can be performed through the [Fast Fourier Transform algorithm](https://en.wikipedia.org/wiki/Fast_Fourier_transform), one of the most important algorithms of history, with a computing cost of $\mathcal{O}(N \log N)$ for data of size $N$ (instead of the naive $N^2$ of matrix-vector multiplication). The discrete equivalent of the Fourier transform is:
$$\tilde{{s}}_{k} = \Delta t \sum_{j=0}^{N-1} \omega^{-jk} {s}_{j} \,, \quad  {s}_{j} = \Delta f \sum_{k=0}^{N-1} \omega^{jk} \tilde{{s}}_{k} \,, \quad \omega = e^{2 i \pi / N} $$

The FFT is implemented in libraries as a mathematical operation, not a 'physical' Fourier transform (with the right units). For instance, `np.fft.fft` has the prefactors $1$ and $1/N$ in the above (see the [doc](https://numpy.org/doc/1.26/reference/routines.fft.html)).

Useful relations are $\Delta t \Delta f = 1/N$ and $\Delta f = 1/T$, with $T$ the total duration.
The DFT also erases the value of the starting time, taking only a 1-dimensional vector as input (the first sample is put at $t=0$).

It is important to ensure the [Nyquist-Shannon criterion](https://en.wikipedia.org/wiki/Nyquist%E2%80%93Shannon_sampling_theorem): we have to choose a sampling frequency $f_s = 1/\Delta t$ such that
$$ \tilde{s}(f) \simeq 0 \quad \text{for} \quad f > f_{\rm nyq} = f_s / 2$$

The output of the DFT (and `np.fft.fft` in particular) gives the negative frequencies in the second half of the vector, $\tilde{s}_k$ for $k=N/2+1, \dots,N-1$. For a real signal, `np.fft.rfft` does the selection automatically.

## An artificial signal

As a first look, let's consider an artificial signal built with a few superposed sine functions.

Let's start by creating an artificial signal made of 3 sinusoidal waves.
We choose the sampling frequency $f_s$ much higher than the maximum frequency, so the Nyquist-Shannon criterion is respected (more on this in the next section).

In [ ]:
# -- Define the sample rate and time array
fs = 1024  # Hz
dt = 1./fs
time = np.arange(0., 1., dt)

# -- Define 3 sine waves
sine1 = np.sin(2*np.pi*23*time)
sine2 = np.sin(2*np.pi*42*time)
sine3 = np.sin(2*np.pi*123*time)

# -- Add the components together, with arbitrary weights
data = 0.8*sine1 + 1.5*sine2 + 0.4*sine3

# -- Plot the signal in the time domain
plt.figure()
plt.plot(time, data)
plt.xlabel('Time(s)')
plt.title('Artificial data in the time domain')

We can now compute the Fourier transform.
The function [rfftfreq](https://numpy.org/doc/1.26/reference/generated/numpy.fft.rfftfreq.html) returns the frequencies on which the FFT will be evaluated.
Here we plot the spectrum, formed by the squared norm $\left| \tilde{s}(f) \right|^2$.
As explained in the previous section, we limit the plot to the range $[0, \frac{f_s}{2}]$.

In [ ]:
# Apply the real Fast Fourier Transform (rfft)
data_freq = dt * np.fft.rfft((data))
frequencies = np.fft.rfftfreq(len(data), d=dt)

plt.plot(frequencies, np.abs(data_freq)**2)
plt.xlim(0, fs/2)
plt.xlabel('Frequency (Hz)')
plt.title("Artificial data in the Frequency Domain")

We can see that the representation of this signal in the frequency domain is much simpler: it's just the 3 values corresponding to the 3 frequencies with a height showing the importance of each component.

## Sampling frequency and the Nyquist Frequency

When we create a discrete representation of a process that is physically understood in the continuous-time limit, we are in some sense erasing some information since we cannot capture features at a resolution in time higher than $dt$. Below, we will illustrate the Nyquist-Shannon criterion that gives the crucial assumption that allows us to use such a representation.

In the following, we illustrate this process with a toy example for a signal: a simple sinusoidal signal at a fixed $f_0=20Hz$, on a time interval [0,1]s (we will represent the continuous-time limit by a very fine time-grid).

We will assume that we only consider signals that are periodic on the interval [0,1]s, which will simplify the use of the DFT. We will sample the continuous process $s(t)$ at different sampling rates $f_s$; for each sampling rate, we will consider the signal formed by the Fourier series with the same number of basis elements as the sampled data.

Concretely, if we have $N$ data points $s_0,\dots,s_{N-1}$, we assume that the continuous-time signal rebuilt from this discrete data is

$$ s_{\rm sampled}(t) = \sum_{j=0}^{N/2} a_j\cos(2\pi j \times t) $$

with coefficients $a_j$ that are related to the DFT coefficients, see 'Trigonometric interpolation polynomial' [here](https://en.wikipedia.org/wiki/Discrete_Fourier_transform#Trigonometric_interpolation_polynomial).

In order to have always integer division of the full interval, we consider sampling rates that are powers of 2, [4Hz, 8Hz, ...].

In [ ]:
# "Continuous time", simply a vector of times with a small dt
continuous_time = np.arange(0, 1, 1./2**10)
continuous_time.shape, continuous_time[1] - continuous_time[0]

In [ ]:
# Generate our true, "continuous" sinusoidal signal
f0 = 20
cos20 = np.cos(2*np.pi*f0*continuous_time)

Let's now show the effect of different sampling frequency.
Play with the sampling frequency $f_s$ and see the sampled signal.
In particular, see what happens when $f_s$ becomes larger than $2f_0$.

> ⚠️ **Warning**: The cell below uses interactive plot which is not displayed correctly on GitHub but works well on Colab or when running locally.
> If you experience problems, comment out the `interact` function and call the `plot_sampling` function directly.

In [ ]:
def coeff(j,N):
    if j==0 or j==N//2:
        return 1
    else:
        return 2
def trigo_polynomial(data, dt, time):
    N = len(data)
    data_rfft = dt * np.fft.rfft(data)
    df = 1/(N * dt)
    return df * np.real(np.sum(np.vstack(np.array([data_rfft[j] * coeff(j,N) * np.cos(2*np.pi*time * j) for j in range(0,N//2+1)])), axis=0))

def plot_sampling(log2fs):
    fs = 2**log2fs
    dt = 1/fs
    T = 1.
    df = 1/T
    sampling_time = np.arange(0, T, dt)
    N = len(sampling_time)
    sampled_cos20 = np.cos(2*np.pi*f0*sampling_time)
    continuous_sampled_signal = trigo_polynomial(sampled_cos20, dt, continuous_time)
    fig, ax = plt.subplots(1,1, figsize=[16,4])
    ax.plot(continuous_time, cos20, label='True signal', color='C0')
    ax.scatter(sampling_time, sampled_cos20, marker='o', label=f'Discrete samples ($f_s = {fs}Hz$)', color='C1')
    ax.plot(continuous_time, continuous_sampled_signal, label=f'Sampled signal', color='C1')
    ax.legend(loc='upper right')
    ax.set_xlabel('t (s)')

interact(
    plot_sampling,
    log2fs = widgets.FloatSlider(value=4, min=1, max=10, step=1, description='$log_2(f_s)$')
)

We can see that if the sampling frequency $f_s$ is too low, we can't reconstruct the original signal.
This is known as [the Shannon-Nyquist theorem](https://en.wikipedia.org/wiki/Nyquist%E2%80%93Shannon_sampling_theorem): if the signal has frequency content up to $f_{max}$, the sampling frequency must be $f_s > 2 \times f_{max}$. This $f_{max}$ is also called the Nyquist frequency. We worked with a signal that is a single frequency here, but this logic would also apply to signals that have a continuous spectrum.

You can also see it the other way around: given the sampling frequency $f_s$, you can't capture features whose frequency if higher than $\frac{f_s}{2}$; this is also called aliasing.

## Fourier transform of a gravitational wave signal

Let's now try on a realistic GW signal.
Following what we did in [this tutorial](../02_Generating_Waveforms/Tuto_2.1_Generating_waveforms.ipynb), we know we can simulate a signal, in that case using a waveform approximant that is natively built in the time domain, `IMRPhenomT`.

In [ ]:
from pycbc.waveform import get_td_waveform, get_fd_waveform

mass1 = 10
mass2 = 15
distance = 100

fs = 4096
f_lower = 30
hp, hc = get_td_waveform(
    approximant='IMRPhenomT',
    mass1=mass1,
    mass2=mass2,
    distance=distance,
    delta_t=1.0/fs,
    f_lower=f_lower
)

# Keep the PyCBC TimeSeries for metadata such as sample_times and delta_t,
# but also make a plain NumPy array for NumPy FFT operations below.
time = hp.sample_times
hp_array = np.asarray(hp)

plt.plot(time, hp_array, label='Plus Polarization')
plt.xlabel('Time (s)')
plt.ylabel('GW Strain')
plt.legend()
plt.title('Whole waveform')

As explained previously, we can also use waveform models that are built directly in the Fourier domain, such as `IMRPhenomXAS`.
Let's do this with the same physical parameters.

In [ ]:
hpf, hcf = get_fd_waveform(
    approximant='IMRPhenomXAS',
    mass1=mass1,
    mass2=mass2,
    distance=distance,
    delta_f=0.125,
    f_lower=f_lower,
    f_ref=800.
)

plt.loglog(hpf.sample_frequencies, np.abs(hpf), label='Plus Polarization')
plt.xlabel('Frequency [Hz]')
plt.ylabel(r'$|\tilde{h}(f)|$ [1/Hz]')
plt.xlim([f_lower, fs/2])
plt.legend()
plt.show()

Surely, if we compute the FT of the time-domain signal, we must find the same spectrum.
Let's try to apply what we did in the beginning of the tutorial.

First, is our sampling frequency high enough? We see that the spectrum of the gravitational wave signal is decaying exponentially at high frequencies; this will be enough in practice, we will be satisfied with the signal frequency content above $f_s/2$ being non-zero but negligible.

We can now compute the Fourier transform of the time-domain waveform:

In [ ]:
# -- Apply the real Fast Fourier Transform (rfft)
dt = hp.delta_t
hp_fft = dt * np.fft.rfft(hp_array)
freq_fft = np.fft.rfftfreq(len(hp_array), d=dt)

plt.loglog(freq_fft, np.abs(hp_fft))
plt.xlim(f_lower/2, fs/2)
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$|\tilde{h}(f)|$ [1/Hz]')
plt.title("Direct FFT of time-domain waveform")

This doesn't look quite correct! We have what is called [Gibbs oscillations](https://en.wikipedia.org/wiki/Gibbs_phenomenon).

The problem is that the FFT works under the assumption that our data are periodic, which means that the edges of our data look like discontinuities when transformed. The post-merger ringdown signal decays exponentially with time, but the waveform has a sharp edge at the start.

We need to apply a window function to our time-domain data before transforming. We will use a [Tukey window](https://docs.scipy.org/doc/scipy-1.13.1/reference/generated/scipy.signal.windows.tukey.html) as implemented in `scipy`, which gives us a smooth transition from 0 to 1 (the profile of the transition is simply a cosine).

In [ ]:
# The window forces the signal to zero at the beginning and end.
# This must be done before applying a Fourier transform.
# The parameter alpha is the fraction of the time interval to be tapered.
# For convenience, we also zero-pad the waveform.

hp_pad = np.concatenate((hp_array, np.zeros(len(hp_array), dtype=float)))
time_pad = np.arange(len(hp_pad)) * dt + float(time[0])

window = scipy.signal.windows.tukey(len(hp_pad), alpha=0.2)
hp_windowed = window * hp_pad

plt.plot(time_pad, hp_pad, label='Zero-padded waveform')
plt.plot(time_pad, hp_windowed, label='Windowed waveform')
plt.xlabel('Time (s)')
plt.ylabel('Strain')
plt.legend()

Repeating the Fourier transform steps, we see that the Gibbs phenomenon is suppressed, and we have a more sensible behaviour at low and high frequencies. We can also compare our FFT of `PhenomT` to `PhenomXAS`.

In [ ]:
# Apply the real Fast Fourier Transform (rfft)
dt = hp.delta_t
hp_w_fft = dt * np.fft.rfft(hp_windowed)
freq_w_fft = np.fft.rfftfreq(len(hp_windowed), d=dt)

plt.loglog(hpf.sample_frequencies, np.abs(hpf), label='FD native waveform')
plt.loglog(freq_fft, np.abs(hp_fft), label='TD waveform + FFT, no window')
plt.loglog(freq_w_fft, np.abs(hp_w_fft), label='TD waveform + FFT, with window')
plt.xlim(5., fs/2)
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$|\tilde{h}(f)|$ [1/Hz]')
plt.title("FFT with window")
plt.legend()

## The inverse Discrete Fourier Transform

We now consider the inverse transformation: starting from Fourier-domain data to obtain time-domain data. This is very similar in practice, with the same requirement to window the Fourier-domain data before doing the IFFT.

We do here the IFFT of the FFT, to check for consistency (normalizations are always tricky!).

In [ ]:
# The input of irfft is of length N/2 + 1.
# We pass n=N explicitly so that the output length matches the padded signal.
dt = hp.delta_t
N = len(hp_windowed)
df = 1 / (N * dt)
hp_w_ifft = (1.0 / dt) * np.fft.irfft(hp_w_fft, n=N)

In [ ]:
# The IFFT of the FFT gives back the same windowed, padded time-domain signal.
# It should be compared against hp_windowed, not the original unpadded waveform.

plt.plot(time_pad, hp_windowed, label='Windowed, padded h')
plt.plot(time_pad, hp_w_ifft, '--', label='IFFT[FFT[windowed, padded h]]')
plt.xlabel('Time (s)')
plt.ylabel('Strain')
plt.legend()

## For reference: PyCBC and GWpy interfaces

We should stress something: conventions are difficult! Many details such as prefactors, keeping track of times, etc. must be handled. Using pre-existing frameworks allows us to use battle-tested tools and stay within consistent conventions.

In **GWpy**, the class `TimeSeries` has a [method](https://gwpy.github.io/docs/2.0.4/api/gwpy.timeseries.TimeSeries.html#gwpy.timeseries.TimeSeries.fft) `fft()`; note that it follows a different convention from ours.
- Our convention, from data vector $\texttt{d}$: $\tilde{\texttt{d}} = dt \times \texttt{FFT}(\texttt{d})$, where $\texttt{FFT}$ is the direct sum without any normalization, as computed by $\texttt{np.fft.rfft}$.
- GWpy, from a `TimeSeries` $\texttt{ts}$: $\texttt{ts.fft}() = 2/N \times \texttt{FFT(ts)} = 2 df * dt \times \texttt{FFT(ts)}$, so there is an extra factor $2df$.

In **PyCBC**, the classes `TimeSeries` and `FrequencySeries` have the methods `to_frequencyseries()` and `to_timeseries()` (see [this page](https://pycbc.org/pycbc/latest/html/fft.html#performing-ffts-in-pycbc)); the Fourier transform implemented by `to_frequencyseries()` had the same convention as us. `TimeSeries` also has a method `taper_timeseries()` to perform windowing. The module `pycbc.fft` also has functions like `fft()` (see e.g. [this page](https://pycbc.org/pycbc/latest/html/fft.html)).

## Fourier transform of real LVK data

Let's now see the FFT in action on real LVK data.
We start by loading data around the event GW190412.

In [ ]:
from gwosc.datasets import event_gps
from gwpy.timeseries import TimeSeries

gps = event_gps('GW190412')
segment = (int(gps) - 5, int(gps) + 5)

# This cell requires internet access unless the data are already cached.
ldata = TimeSeries.fetch_open_data('L1', *segment, verbose=True, cache=True)

We can calculate the Fourier transform of our GWpy `TimeSeries` using the [fft()](https://gwpy.readthedocs.io/en/v3.0.14/api/gwpy.timeseries.TimeSeries/?highlight=fourier#gwpy.timeseries.TimeSeries.fft) method:

In [ ]:
fft = ldata.fft()
print(fft)

The result is a [`FrequencySeries`](https://gwpy.readthedocs.io/en/v3.0.14/api/gwpy.frequencyseries.FrequencySeries/#gwpy.frequencyseries.FrequencySeries), with complex amplitude, representing the amplitude and phase of each frequency in our data.
We can use `abs()` to extract the amplitude and plot it:

In [ ]:
plot = fft.abs().plot(xscale="log", yscale="log")
plot.show(warn=False)

As before, we need to apply a window function to our time-domain data before transforming, for which use again the Tukey window.

In [ ]:
window_tukey = scipy.signal.windows.tukey(len(ldata), alpha=0.2)
lwin_tukey = ldata * window_tukey

Let's try our transform again and see what we get

In [ ]:
fftamp_tukey = lwin_tukey.fft().abs()
plot = fftamp_tukey.plot(xscale="log", yscale="log")
plot.show(warn=False)

This looks a little more like what we expect for the amplitude spectral density of a gravitational-wave detector.

## Question 1

Repeat the FFT exercise for GW190412, but this time using a `hann` window. Do you see any differences? Also plot the hann window function and the tukey window function on top of each other, with the x-axis being the time (ldata.times).

In [ ]:
window_hann = scipy.signal.windows.hann(len(ldata))
lwin_hann = ldata * window_hann
fftamp_hann = lwin_hann.fft().abs()

plot = fftamp_hann.plot(xscale="log", yscale="log")
plot.show(warn=False)

In [ ]:
window_hann = scipy.signal.windows.hann(len(ldata))
window_tukey = scipy.signal.windows.tukey(len(ldata), alpha=0.2)

# Use time relative to the first sample so the window shapes are easy to read.
relative_time = (ldata.times - ldata.times[0]).to_value('s')

plt.plot(relative_time, window_hann, label="Hann window")
plt.plot(relative_time, window_tukey, label="Tukey window")
plt.xlabel('Time from segment start (s)')
plt.ylabel('Window amplitude')
plt.legend()